In [3]:
import numpy as np
import tensorflow as tf
import os
import glob
import cv2 
from sklearn.model_selection import train_test_split
import sys
from io import StringIO
from collections import Counter
import struct # Required for reading MNIST binary files

# Capture stdout
old_stdout = sys.stdout
sys.stdout = StringIO()

# --- Config ---
# Assuming MNIST files are directly inside the 'MNIST' folder
DATA_PATH = "./MNIST" 
EXPORT_NAME = "hu_moment_neuron_config"

NUM_FEATURES = 7 

# Define paths to the binary files
TRAIN_IMAGES_PATH = os.path.join(DATA_PATH, "train-images.idx3-ubyte")
TRAIN_LABELS_PATH = os.path.join(DATA_PATH, "train-labels.idx1-ubyte")
TEST_IMAGES_PATH = os.path.join(DATA_PATH, "t10k-images.idx3-ubyte")
TEST_LABELS_PATH = os.path.join(DATA_PATH, "t10k-labels.idx1-ubyte")


# --- 1. MNIST BINARY LOADER FUNCTIONS ---

def load_mnist_images(file_path):
    """ Loads images from MNIST binary file format (.idx3-ubyte) """
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"MNIST image file not found: {file_path}")
        
    with open(file_path, 'rb') as f:
        # Read magic number (4 bytes), number of items (4 bytes), rows (4 bytes), cols (4 bytes)
        magic, num, rows, cols = struct.unpack('>IIII', f.read(16))
        
        # Check magic number (should be 2051)
        if magic != 2051:
            raise ValueError(f"Invalid MNIST image magic number: {magic}")
            
        # Read all image data (num * rows * cols bytes)
        images = np.fromfile(f, dtype=np.uint8)
        images = images.reshape(num, rows, cols)
        return images

def load_mnist_labels(file_path):
    """ Loads labels from MNIST binary file format (.idx1-ubyte) """
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"MNIST label file not found: {file_path}")

    with open(file_path, 'rb') as f:
        # Read magic number (4 bytes), number of items (4 bytes)
        magic, num = struct.unpack('>II', f.read(8))
        
        # Check magic number (should be 2049)
        if magic != 2049:
            raise ValueError(f"Invalid MNIST label magic number: {magic}")
            
        # Read all label data (num bytes)
        labels = np.fromfile(f, dtype=np.uint8)
        return labels


# --- 2. HU MOMENT EXTRACTION ---

def extract_hu_moments(image):
    """ Calculates the 7 Hu Moments and applies log transformation. """
    # Image must be single-channel (grayscale), which MNIST is.
    
    # 1. Calculate Moments (Central Moments)
    # Note: Hu Moments expects the image to be normalized/thresholded for best results, 
    # but for consistent feature extraction, we process the raw image moments.
    moments = cv2.moments(image)
    
    # 2. Calculate Hu Moments (7 features)
    hu_moments_raw = cv2.HuMoments(moments).flatten()
    
    # 3. Apply standard log transformation for scale and rotation invariance
    # Transformation: -sign(Hu_i) * log10(abs(Hu_i))
    hu_moments = -np.sign(hu_moments_raw) * np.log10(np.abs(hu_moments_raw) + 1e-7)
    
    return hu_moments[:NUM_FEATURES].astype(np.float32)


def prepare_dataset():
    # Load all data using the custom loaders
    print("Loading MNIST images and labels...")
    train_images = load_mnist_images(TRAIN_IMAGES_PATH)
    train_labels = load_mnist_labels(TRAIN_LABELS_PATH)
    test_images = load_mnist_images(TEST_IMAGES_PATH)
    test_labels = load_mnist_labels(TEST_LABELS_PATH)
    
    # Combine training and testing sets for standardized feature extraction
    all_images = np.concatenate([train_images, test_images])
    all_labels = np.concatenate([train_labels, test_labels])
    
    X_raw, y = [], []
    
    # Binary Classification: '0' vs 'Not 0'
    print("Extracting Hu Moments and assigning binary labels ('0' vs 'Not 0')...")
    
    for img, label in zip(all_images, all_labels):
        
        # Target Class (1): Digit '0'
        # Other Class (0): All other digits
        label_bin = 1.0 if label == 0 else 0.0
        
        moments = extract_hu_moments(img)
        
        X_raw.append(moments)
        y.append(label_bin)

    X_raw = np.array(X_raw, dtype=np.float32)
    y = np.array(y, dtype=np.float32)
    
    print(f"Total processed samples: {len(X_raw)}.")
    
    # Re-split data to ensure train/test sets are cleanly separated after extraction
    # Using the same fixed seed (42) for reproducibility
    return train_test_split(X_raw, y, test_size=0.2, random_state=42)

# --- 3. EXPORT TO C (Standard logic adjusted for 7 features) ---

def export_c(model, filename, mean, std):
    w, b = model.layers[0].get_weights()
    w_flat = w.flatten()
    b_val = b[0]
    
    with open(f"{filename}.h", "w") as f:
        f.write(f"#ifndef {filename.upper()}_H\n#define {filename.upper()}_H\n\n")
        f.write(f"#define NUM_FEATURES {NUM_FEATURES}\n")
        f.write("float neuron_predict(float *features);\\n#endif\n")
        
    with open(f"{filename}.c", "w") as f:
        f.write(f'#include "{filename}.h"\n#include <math.h>\n\n')
        
        f.write(f"static const float W[{NUM_FEATURES}] = {{\n    ")
        for i, val in enumerate(w_flat):
            f.write(f"{val:.8f}f, ")
            if (i+1)%5==0 and i < len(w_flat) - 1: f.write("\n    ")
        f.write(f"\n}};")
        
        f.write(f"\n\nstatic const float B = {b_val:.8f}f;\n\n")
        
        f.write("float neuron_predict(float *x) {\n")
        f.write("    float z = B;\n")
        f.write(f"    for(int i=0; i<{NUM_FEATURES}; i++) z += x[i] * W[i];\n")
        f.write("    return 1.0f / (1.0f + expf(-z));\n")
        f.write("}\n")
    
    sys.stdout = old_stdout
    print("\n" + "="*70)
    print("CRITICAL: Copy the following normalization constants to your C application.")
    print("="*70)
    
    print(f"static const float HU_MEAN[NUM_FEATURES] = {{")
    print("    ", end="")
    for i in range(NUM_FEATURES):
        print(f"{mean[i]:.4f}f, ", end="")
        if (i + 1) % 5 == 0 and i < NUM_FEATURES - 1:
            print("\n    ", end="")
    print(f"\n}}; // {NUM_FEATURES} elements")
    
    print(f"\nstatic const float HU_STD[NUM_FEATURES] = {{")
    print("    ", end="")
    for i in range(NUM_FEATURES):
        safe_std = max(std[i], 1e-6) 
        print(f"{safe_std:.4f}f, ", end="")
        if (i + 1) % 5 == 0 and i < NUM_FEATURES - 1:
            print("\n    ", end="")
    print(f"\n}}; // {NUM_FEATURES} elements")
    print("="*70)
    sys.stdout = StringIO()


# --- MAIN EXECUTION ---
if __name__ == "__main__":
    
    sys.stdout = old_stdout
    
    # Load and prepare data (This function performs extraction and train/test split)
    X_train_raw, X_test_raw, y_train, y_test = prepare_dataset()
    
    if len(X_train_raw) == 0:
        print("No valid features extracted. Check 'MNIST' folder path and file contents.")
    else:
        print(f"\nTraining samples: {len(X_train_raw)}")
        print(f"Testing samples: {len(X_test_raw)}")
        
        # 4. Normalize
        X_mean = np.mean(X_train_raw, axis=0, dtype=np.float32)
        X_std = np.std(X_train_raw, axis=0, dtype=np.float32)
        
        X_train_norm = (X_train_raw - X_mean) / (X_std + 1e-7)
        X_test_norm = (X_test_raw - X_mean) / (X_std + 1e-7)
        
        # --- CLASS WEIGHT CALCULATION ---
        counts = Counter(y_train)
        total_samples = len(y_train)
        
        # MNIST '0' is usually 10% of the data, so weighting is critical
        weight_0 = total_samples / (2 * counts.get(0.0, 1)) # Weight for 'Not Zero' (~90%)
        weight_1 = total_samples / (2 * counts.get(1.0, 1)) # Weight for 'Zero' (~10%)
        class_weights = {0: weight_0, 1: weight_1}
        print(f"\nClass Weights: {class_weights}")


        # 5. Training Keras Model
        print(f"Starting Keras Training...")
        
        model = tf.keras.Sequential([
            tf.keras.layers.Input(shape=(NUM_FEATURES,)),
            tf.keras.layers.Dense(1, activation='sigmoid', dtype=tf.float32)
        ])
        
        model.compile(
            optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
            loss='binary_crossentropy',
            metrics=['accuracy']
        )
        
        history = model.fit(
            X_train_norm.astype(np.float32), y_train.astype(np.float32), 
            epochs=50, 
            batch_size=32, 
            class_weight=class_weights, 
            validation_data=(X_test_norm.astype(np.float32), y_test.astype(np.float32)),
            verbose=1
        )
        
        # 6. Export Weights, Bias, and Normalization Constants
        sys.stdout = StringIO() 
        export_c(model, EXPORT_NAME, X_mean, X_std)
        
        sys.stdout = old_stdout 
        print("Model configuration and normalization constants successfully exported.")

Loading MNIST images and labels...
Extracting Hu Moments and assigning binary labels ('0' vs 'Not 0')...
Total processed samples: 70000.

Training samples: 56000
Testing samples: 14000

Class Weights: {0: 0.5551149881046789, 1: 5.0359712230215825}
Starting Keras Training...
Epoch 1/50
1750/1750 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step - accuracy: 0.6552 - loss: 0.5277 - val_accuracy: 0.7298 - val_loss: 0.5121
Epoch 2/50
1750/1750 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.7456 - loss: 0.4456 - val_accuracy: 0.7751 - val_loss: 0.4552
Epoch 3/50
1750/1750 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.7777 - loss: 0.4188 - val_accuracy: 0.7860 - val_loss: 0.4394
Epoch 4/50
1750/1750 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.7881 - loss: 0.4030 - val_accuracy: 0.8061 - val_loss: 0.4141
Epoch 5/50
1750/1750 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.8016 - loss: 0.3919 - val_accuracy: 0.8011 - val_loss: 0.4151
Epoch 6/50
1750/1750 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.8041 - 

In [4]:

# NOTE: This script assumes 'model', 'X_test_norm', and 'y_test' 
# are available in the current environment after running the training script.

import numpy as np
import tensorflow as tf
from sklearn.metrics import confusion_matrix, accuracy_score

# --- 1. EXTRACT DEPLOYMENT CONSTANTS (32-bit forced) ---

# Extract weights and bias directly from the trained Keras model (default 64-bit NumPy floats)
W_DEPLOYED_64, B_DEPLOYED_64 = model.layers[0].get_weights()

# CRITICAL: Force constants to 32-bit float (np.float32), 
# matching the precision used when they are written to the C file.
W_DEPLOYED = W_DEPLOYED_64.flatten().astype(np.float32)
B_DEPLOYED = B_DEPLOYED_64[0].astype(np.float32)

# --- 2. LOCAL NEURON IMPLEMENTATION (C 32-bit Simulation) ---

def neuron_predict_c_sim(features_32bit):
    """ 
    Simulates the single neuron prediction function defined in C,
    using 32-bit weights and ensuring 32-bit NumPy operations.
    """
    
    # Linear combination (W * X + B)
    # Ensure all intermediate operations remain 32-bit
    z = B_DEPLOYED + np.dot(features_32bit, W_DEPLOYED).astype(np.float32)
    
    # Sigmoid Activation
    return 1.0 / (1.0 + np.exp(-z))

# --- 3. EXECUTION AND VALIDATION ---

if 'model' not in locals() or 'X_test_norm' not in locals() or 'y_test' not in locals():
    print("❌ ERROR: Required variables (model, X_test_norm, y_test) not found.")
    print("Ensure this code runs immediately after the Keras training cell.")
    exit()

# Prepare 32-bit input data for simulation
X_test_norm_32 = X_test_norm.astype(np.float32)
y_test_bin = y_test.astype(int)

print("\n--- Local C 32-bit Simulation Verification ---")

# 3a. Predict using the C-simulated function
probabilities_c_sim = neuron_predict_c_sim(X_test_norm_32)

# 3b. Predict using the Keras model (baseline comparison)
# Note: Providing 32-bit input, but Keras operations might be higher precision.
probabilities_keras = model.predict(X_test_norm_32, verbose=0).flatten()

# Convert probabilities to binary prediction (0 or 1)
predictions_c_sim = (probabilities_c_sim > 0.5).astype(int)

# Calculate metrics
acc_c_sim = accuracy_score(y_test_bin, predictions_c_sim)
cm = confusion_matrix(y_test_bin, predictions_c_sim)

print("-" * 50)
print(f"Total Test Samples: {len(X_test_norm_32)}")
print(f"MCU Simulated 32-bit Accuracy: {acc_c_sim * 100:.4f}%")
print(f"Keras Validation Accuracy (32-bit input baseline): {accuracy_score(y_test_bin, (probabilities_keras > 0.5).astype(int)) * 100:.4f}%")
print("-" * 50)
print("Confusion Matrix (C Sim):")
print(f"| TN (Pred 0, Actual 0): {cm[0, 0]} | FP (Pred 1, Actual 0): {cm[0, 1]} |")
print(f"| FN (Pred 0, Actual 1): {cm[1, 0]} | TP (Pred 1, Actual 1): {cm[1, 1]} |")
print("-" * 50)

# Final check ensures numerical stability (optional, but good practice)
if np.allclose(W_DEPLOYED, W_DEPLOYED_64.flatten(), atol=1e-5) and np.allclose(B_DEPLOYED, B_DEPLOYED_64[0], atol=1e-5):
    print("✅ C Constants (W, B) extracted match the Keras model perfectly (within float tolerance).")
else:
    print("⚠️ WARNING: C Constants (W, B) extraction mismatch.")


--- Local C 32-bit Simulation Verification ---


C:\Users\ng822\AppData\Local\Temp\ipykernel_21488\4013585475.py:31: RuntimeWarning: overflow encountered in exp
  return 1.0 / (1.0 + np.exp(-z))


--------------------------------------------------
Total Test Samples: 14000
MCU Simulated 32-bit Accuracy: 84.5214%
Keras Validation Accuracy (32-bit input baseline): 84.5214%
--------------------------------------------------
Confusion Matrix (C Sim):
| TN (Pred 0, Actual 0): 10609 | FP (Pred 1, Actual 0): 2048 |
| FN (Pred 0, Actual 1): 119 | TP (Pred 1, Actual 1): 1224 |
--------------------------------------------------
✅ C Constants (W, B) extracted match the Keras model perfectly (within float tolerance).
